# Hyperliquid Portfolio & PnL Dashboard

Tracks account balances, open perp positions, and realized/unrealized PnL including trading fees and signed perp funding from Hyperliquid data cached in CSV. Calculates portfolio performance and risk metrics (including VaR and historical volatility), attributes PnL by asset, and compares account performance with BTC through charts and summary statistics.


In [ ]:
"""Reusable API, parsing, and history-persistence helpers."""
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from types import SimpleNamespace
from time import sleep

pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

API = "https://api.hyperliquid.xyz/info"
ADDR = "0x7532dA34edA8f3e383bC480d50e7A7d3b12c07f2"  # Account to analyze
HISTORY_COLUMNS = [
    "ts", "accountValue", "source", "asset", "balance", "open_position_value",
    "unrealized_pnl", "realized_pnl",
]


def post(payload):
    """Send one request to Hyperliquid's public info endpoint."""
    response = requests.post(API, json=payload, timeout=15)
    response.raise_for_status()
    return response.json()


def as_float(value, default=0.0):
    """Convert API number strings safely, returning a default for missing values."""
    return float(value) if value not in (None, "") else default


def spot_balance_usd(user=ADDR):
    """Value spot balances using available spot markets and USD stablecoin anchors."""
    spot_state = post({"type": "spotClearinghouseState", "user": user})
    metadata = post({"type": "spotMeta"})
    mids = post({"type": "allMids", "dex": ""})

    tokens = {token.get("index"): token.get("name") for token in metadata.get("tokens", [])}
    # Build conversion edges from each spot market's mid. The market's @index key
    # is its asset index in allMids; token indices identify its base and quote.
    rates = {}
    for market in metadata.get("universe", []):
        token_ids = market.get("tokens", [])
        if len(token_ids) != 2:
            continue
        market_index = market.get("index")
        mid = as_float(mids.get(f"@{market_index}"), default=0.0)
        if mid <= 0:
            continue
        base, quote = (tokens.get(token_id) for token_id in token_ids)
        if not base or not quote:
            continue
        rates.setdefault(base, []).append((quote, mid))
        rates.setdefault(quote, []).append((base, 1.0 / mid))

    # Stablecoins used as USD anchors. USDC is the primary Hyperliquid quote asset.
    usd_prices = {coin: 1.0 for coin in ("USDC", "USDT", "USDE", "DAI", "USDH")}
    # Iteratively propagate prices across connected markets.
    changed = True
    while changed:
        changed = False
        for coin, edges in rates.items():
            if coin in usd_prices:
                continue
            for quote, coin_in_quote in edges:
                if quote in usd_prices:
                    usd_prices[coin] = coin_in_quote * usd_prices[quote]
                    changed = True
                    break

    total_usd, priced, excluded = 0.0, [], []
    for balance in spot_state.get("balances", []):
        amount = as_float(balance.get("total"))
        if not amount:
            continue
        coin = balance.get("coin", "")
        price = usd_prices.get(coin)
        if price is None or price <= 0:
            excluded.append((coin, amount))
            continue
        value_usd = amount * price
        total_usd += value_usd
        priced.append((coin, amount, price, value_usd))
    return total_usd, priced, excluded
def load_saved_history(csv_path):
    """Load the local history CSV, or return an empty frame with its schema."""
    if not csv_path.exists():
        return pd.DataFrame(columns=HISTORY_COLUMNS)
    saved = pd.read_csv(csv_path, parse_dates=["ts"])
    saved["ts"] = pd.to_datetime(saved["ts"], utc=True).dt.tz_localize(None)
    return saved


def fetch_all_time_history(user):
    """Fetch the account's allTime API history for an initial/full refresh."""
    portfolio = post({"type": "portfolio", "user": user})
    periods = {
        item[0]: item[1]
        for item in portfolio
        if isinstance(item, (list, tuple)) and len(item) == 2 and isinstance(item[1], dict)
    }
    history = periods.get("allTime", {}).get("accountValueHistory", [])
    if not history:
        raise ValueError("No allTime account value history was returned.")
    return history


def fetch_perp_states(user):
    """Fetch the clearinghouse state from main Hyperliquid and every listed perp DEX."""
    dexes = post({"type": "perpDexs"})
    names = [""] + [dex["name"] for dex in dexes if dex and dex.get("name")]
    return {
        dex: post({"type": "clearinghouseState", "user": user, "dex": dex})
        for dex in names
    }


def fetch_user_fills(user, max_pages=5):
    """Fetch up to 10,000 most recent fills, paging backward in time."""
    fills = []
    end_ms = int(pd.Timestamp.now(tz="UTC").timestamp() * 1000)
    for _ in range(max_pages):
        batch = post({
            "type": "userFillsByTime", "user": user, "startTime": 0,
            "endTime": end_ms, "aggregateByTime": True,
        })
        if not isinstance(batch, list) or not batch:
            break
        fills.extend(batch)
        times = [int(fill["time"]) for fill in batch if fill.get("time") is not None]
        if len(batch) < 2000 or not times or len(fills) >= 10000:
            break
        next_end_ms = min(times) - 1
        if next_end_ms >= end_ms:
            break
        end_ms = next_end_ms
    return fills


def fetch_user_funding(user, start_ms=0):
    """Fetch signed USDC funding transfers, paging through 500-event responses."""
    end_ms = int(pd.Timestamp.now(tz="UTC").timestamp() * 1000)
    cursor = max(0, int(start_ms))
    events = []
    page = 0
    while cursor <= end_ms:
        batch = post({
            "type": "userFunding", "user": user,
            "startTime": cursor, "endTime": end_ms,
        })
        if not isinstance(batch, list) or not batch:
            break
        events.extend(batch)
        page += 1
        times = [int(event["time"]) for event in batch if event.get("time") is not None]
        if len(batch) < 500 or not times:
            break
        next_start = max(times) + 1
        if next_start <= cursor:
            raise RuntimeError("Funding history pagination did not advance; stopping to avoid duplicate requests.")
        cursor = next_start
        if page % 10 == 0:
            print(f"Fetched {len(events):,} funding records ({page} pages)...")
        # A full page has a higher API weight; throttle to stay within documented limits.
        sleep(2.6)
    return events


def normalize_funding(events):
    """Parse signed USDC funding transfers; negative means paid, positive received."""
    rows = []
    for event in events:
        delta = event.get("delta", {}) or {}
        if event.get("time") is None or delta.get("usdc") is None:
            continue
        rows.append({
            "time": int(event["time"]),
            "coin": str(delta.get("coin") or "Unknown").strip(),
            "fundingPnl": as_float(delta.get("usdc"), default=float("nan")),
        })
    frame = pd.DataFrame(rows, columns=["time", "coin", "fundingPnl"])
    if not frame.empty:
        frame["ts"] = pd.to_datetime(frame["time"], unit="ms", utc=True).dt.tz_localize(None)
        frame["day"] = frame["ts"].dt.normalize()
    return frame


def make_daily_funding_rows(funding_df):
    """Aggregate signed funding by UTC day and perp asset for history and attribution."""
    if funding_df.empty:
        return pd.DataFrame(columns=HISTORY_COLUMNS)
    daily = funding_df.groupby(["day", "coin"], as_index=False)["fundingPnl"].sum()
    daily = daily.rename(columns={"day": "ts", "coin": "asset", "fundingPnl": "realized_pnl"})
    daily["source"] = "funding_daily"
    for column in ["accountValue", "balance", "open_position_value", "unrealized_pnl"]:
        daily[column] = float("nan")
    return daily.reindex(columns=HISTORY_COLUMNS)


def normalize_fills(fills):
    """Create a consistent spot/perp fill table for PnL calculations."""
    rows = [{
        "time": int(fill["time"]),
        "coin": fill.get("coin", ""),
        "assetType": "spot" if str(fill.get("coin", "")).startswith("@") else "perp",
        "dir": fill.get("dir", ""),
        "notional": abs(as_float(fill.get("px")) * as_float(fill.get("sz"))),
        "closedPnl": as_float(fill.get("closedPnl")),
        "fee": as_float(fill.get("fee"), default=float("nan")),
        "feeToken": str(fill.get("feeToken") or "Unknown").strip(),
    } for fill in fills if fill.get("time") is not None]
    frame = pd.DataFrame(rows)
    if not frame.empty:
        frame["ts"] = pd.to_datetime(frame["time"], unit="ms", utc=True)
        frame["day"] = frame["ts"].dt.floor("D").dt.tz_localize(None)
    return frame


def make_api_history_rows(history, saved):
    """Format new API points and skip timestamps already stored in the CSV."""
    frame = pd.DataFrame(history, columns=["ts", "accountValue"])
    frame["ts"] = pd.to_datetime(frame["ts"], unit="ms")
    frame["accountValue"] = frame["accountValue"].astype(float)
    frame["source"] = "portfolio_allTime"
    for column in ["balance", "open_position_value", "unrealized_pnl", "realized_pnl"]:
        frame[column] = float("nan")
    if not saved.empty:
        previous = saved[saved["source"] == "portfolio_allTime"]
        last_ts = previous["ts"].max() if not previous.empty else pd.Timestamp.min
        frame = frame[frame["ts"] > last_ts]
    return frame


def make_live_snapshot(account_value, balance, position_value, unrealized_pnl):
    """Build one current snapshot using the live balance + open-position uPnL formula."""
    return pd.DataFrame([{
        "ts": pd.Timestamp.now(tz="UTC").tz_localize(None),
        "accountValue": account_value,
        "source": "live_snapshot",
        "balance": balance,
        "open_position_value": position_value,
        "unrealized_pnl": unrealized_pnl,
        "realized_pnl": float("nan"),
    }])


def make_daily_realized_rows(realized_df):
    """Aggregate fill PnL by day and asset class for persistent history."""
    daily_frames = []
    if not realized_df.empty:
        for asset_type, source_name in [("perp", "perp_realized_daily"), ("spot", "spot_realized_daily")]:
            daily = realized_df[realized_df["assetType"] == asset_type].groupby("day", as_index=False)["closedPnl"].sum()
            if daily.empty:
                continue
            daily = daily.rename(columns={"day": "ts", "closedPnl": "realized_pnl"})
            daily["source"] = source_name
            for column in ["accountValue", "balance", "open_position_value", "unrealized_pnl"]:
                daily[column] = float("nan")
            daily_frames.append(daily)
    if daily_frames:
        return pd.concat(daily_frames, ignore_index=True)
    return pd.DataFrame(columns=HISTORY_COLUMNS)


def merge_and_save_history(csv_path, *frames):
    """Combine all history sources, keep the latest row per timestamp/source, and save."""
    combined = pd.concat([frame.reindex(columns=HISTORY_COLUMNS) for frame in frames], ignore_index=True)
    combined["ts"] = pd.to_datetime(combined["ts"])
    combined = combined.drop_duplicates(subset=["ts", "source", "asset"], keep="last")
    combined = combined.sort_values("ts").reset_index(drop=True)
    combined.to_csv(csv_path, index=False)
    return combined


def fetch_btc_candles(start_ms, end_ms, interval="1d"):
    """Fetch BTC candles for a millisecond time range and requested interval."""
    return pd.DataFrame(post({
        "type": "candleSnapshot",
        "req": {"coin": "BTC", "interval": interval, "startTime": start_ms, "endTime": end_ms},
    }))


## Account data and history

Fetches current values, loads cached history, and saves new snapshots and realized PnL.


In [ ]:
"""Orchestrate account data collection and local history persistence."""
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = DATA_DIR / "hyperliquid_account_history.csv"
FETCH_FULL_HISTORY = False
saved = load_saved_history(CSV_PATH)

# Fetch allTime API history only on first run or when explicitly requested.
history = fetch_all_time_history(ADDR) if saved.empty or FETCH_FULL_HISTORY else []
history_rows_added = 0

# Current account value uses valued spot balance plus unrealized PnL from every perp DEX.
current_states = fetch_perp_states(ADDR)
balance_total, priced_spot, unpriced_spot = spot_balance_usd(ADDR)
unrealized_pnl_total = sum(
    as_float(item.get("position", {}).get("unrealizedPnl"))
    for state in current_states.values()
    for item in state.get("assetPositions", [])
)
position_value_total = sum(
    abs(as_float(item.get("position", {}).get("positionValue")))
    for state in current_states.values()
    for item in state.get("assetPositions", [])
)
current_account_value = balance_total + unrealized_pnl_total

# Normalize fills once; later cells reuse this frame for stats and attribution.
realized_fills = fetch_user_fills(ADDR)
realized_df = normalize_fills(realized_fills)

# Refresh all funding on first/full refresh; otherwise refetch from the latest saved UTC day.
saved_funding = saved.loc[saved["source"] == "funding_daily"] if not saved.empty else pd.DataFrame()
if FETCH_FULL_HISTORY or saved_funding.empty:
    funding_start_ms = 0
else:
    latest_funding_day = pd.to_datetime(saved_funding["ts"], utc=True).max().normalize()
    funding_start_ms = int(latest_funding_day.timestamp() * 1000)
funding_events = fetch_user_funding(ADDR, start_ms=funding_start_ms)
funding_df = normalize_funding(funding_events)
daily_funding_new = make_daily_funding_rows(funding_df)
print(f"Funding records fetched this run: {len(funding_df):,}")

# Convert each data source to the same CSV schema, then merge and persist it.
new_history = make_api_history_rows(history, saved)
history_rows_added = len(new_history)
live_snapshot = make_live_snapshot(
    current_account_value, balance_total, position_value_total, unrealized_pnl_total
)
daily_realized_new = make_daily_realized_rows(realized_df)
chart_df = merge_and_save_history(
    CSV_PATH, saved, new_history, live_snapshot, daily_realized_new, daily_funding_new
)
funding_history_df = chart_df.loc[
    chart_df["source"] == "funding_daily", ["ts", "asset", "realized_pnl"]
].rename(columns={"asset": "coin", "realized_pnl": "fundingPnl"}).copy()
funding_history_df["ts"] = pd.to_datetime(funding_history_df["ts"], utc=True)


Fetches current values, loads cached history, and saves new snapshots, realized trade PnL, and signed perp funding.


In [ ]:
"""Calculate account, PnL, return-distribution, VaR, and BTC risk metrics."""
def historical_var(pnl_observations, account_value, confidence, min_observations):
    """Estimate loss VaR from the empirical lower-tail daily PnL quantile."""
    sample = pd.to_numeric(pnl_observations, errors="coerce").dropna()
    count = len(sample)
    if count < min_observations or account_value <= 0:
        return {"usd": float("nan"), "pct": float("nan"), "n": count, "quantile_pnl": float("nan")}
    quantile_pnl = float(sample.quantile(1 - confidence))
    loss_usd = max(0.0, -quantile_pnl)
    return {
        "usd": loss_usd,
        "pct": loss_usd / account_value * 100,
        "n": count,
        "quantile_pnl": quantile_pnl,
    }


def historical_volatility(pnl_observations, account_value, annualization_days=365):
    """Estimate daily and annualized volatility from observed daily PnL."""
    sample = pd.to_numeric(pnl_observations, errors="coerce").dropna()
    count = len(sample)
    if count < 2 or account_value <= 0:
        return {"daily_pct": float("nan"), "annualized_pct": float("nan"), "n": count}
    daily_returns_pct = sample / account_value * 100
    daily_vol_pct = float(daily_returns_pct.std(ddof=1))
    return {
        "daily_pct": daily_vol_pct,
        "annualized_pct": daily_vol_pct * (annualization_days ** 0.5),
        "n": count,
    }

def calculate_account_metrics(chart_df, current_account_value, unrealized_pnl_total):
    """Build daily PnL/return series and risk metrics from saved account history."""
    account_series = chart_df.loc[chart_df["source"] == "portfolio_allTime"].set_index("ts")["accountValue"].dropna()
    daily_account = account_series.resample("1D").last().dropna()
    realized_sources = ["perp_realized_daily", "spot_realized_daily", "funding_daily"]
    realized_history = chart_df.loc[chart_df["source"].isin(realized_sources), ["ts", "realized_pnl"]].dropna()
    realized_series = realized_history.groupby("ts")["realized_pnl"].sum().sort_index()
    unrealized_series = chart_df.loc[chart_df["source"] == "live_snapshot"].set_index("ts")["unrealized_pnl"].dropna()
    daily_unrealized = unrealized_series.resample("1D").last().dropna()

    if not realized_series.empty or not daily_unrealized.empty:
        date_min = min(series.index.min() for series in [realized_series, daily_unrealized] if not series.empty)
        date_max = max(series.index.max() for series in [realized_series, daily_unrealized] if not series.empty)
        day_index = pd.date_range(date_min, date_max, freq="1D")
        daily_realized = realized_series.reindex(day_index, fill_value=0.0)
        daily_unrealized = daily_unrealized.reindex(day_index)
    else:
        day_index = pd.DatetimeIndex([])
        daily_realized = pd.Series(dtype=float, index=day_index)
        daily_unrealized = pd.Series(dtype=float, index=day_index)

    realized_avg_day = daily_realized.mean() if not daily_realized.empty else float("nan")
    u_change = daily_unrealized.dropna().diff().dropna()
    u_avg_day = u_change.mean() if not u_change.empty else float("nan")
    net_change = daily_realized.add(daily_unrealized.ffill().diff().fillna(0), fill_value=0).dropna()
    net_avg_day = net_change.mean() if not net_change.empty else float("nan")
    equity_for_returns = daily_account.reindex(net_change.index).ffill()
    daily_return_pct = (net_change / equity_for_returns.shift(1) * 100).replace(
        [float("inf"), -float("inf")], float("nan")
    ).dropna()
    avg_daily_return_pct = daily_return_pct.mean() if not daily_return_pct.empty else float("nan")
    max_daily_gain_pct = daily_return_pct.max() if not daily_return_pct.empty else float("nan")
    max_daily_loss_pct = daily_return_pct.min() if not daily_return_pct.empty else float("nan")
    monthly_return_pct = ((1 + daily_return_pct / 100).groupby(
        daily_return_pct.index.to_period("M")
    ).prod() - 1) * 100
    monthly_return_pct.index = monthly_return_pct.index.to_timestamp()

    cumulative_pnl = daily_realized.cumsum() + daily_unrealized.ffill().fillna(0)
    pnl_dd_usd = (cumulative_pnl - cumulative_pnl.cummax()).min() if not cumulative_pnl.empty else float("nan")
    if not daily_account.empty:
        account_peak = daily_account.cummax()
        account_dd_usd = (daily_account - account_peak).min()
        positive = daily_account > 0
        account_dd_pct = ((daily_account[positive] / account_peak[positive]) - 1).min() * 100 if positive.any() else float("nan")
    else:
        account_dd_usd = account_dd_pct = float("nan")

    live_account = chart_df.loc[chart_df["source"] == "live_snapshot"].set_index("ts")["accountValue"].dropna()
    live_account_daily = live_account.resample("1D").last().dropna()

    # Build observed daily account PnL from saved realized spot/perp results and
    # day-over-day uPnL changes. Missing historical mark-to-market days stay out
    # of the sample instead of being incorrectly treated as zero-PnL days.
    realized_history = chart_df.loc[
        chart_df["source"].isin(["perp_realized_daily", "spot_realized_daily", "funding_daily"]), ["ts", "realized_pnl"]
    ].dropna().copy()
    if not realized_history.empty:
        realized_history["day"] = pd.to_datetime(realized_history["ts"]).dt.normalize()
        account_realized_daily = realized_history.groupby("day")["realized_pnl"].sum()
    else:
        account_realized_daily = pd.Series(dtype=float)
    account_unrealized_daily = chart_df.loc[
        chart_df["source"] == "live_snapshot"
    ].set_index("ts")["unrealized_pnl"].dropna().resample("1D").last().dropna()
    account_unrealized_change = account_unrealized_daily.diff()
    # Only treat consecutive dated snapshots as one-day uPnL changes.
    consecutive_days = account_unrealized_daily.index.to_series().diff().eq(pd.Timedelta(days=1))
    account_unrealized_change = account_unrealized_change[consecutive_days].dropna()
    observed_days = account_realized_daily.index.union(account_unrealized_change.index).sort_values()
    observed_daily_pnl = account_realized_daily.reindex(observed_days, fill_value=0.0).add(
        account_unrealized_change.reindex(observed_days, fill_value=0.0)
    )
    var_95 = historical_var(observed_daily_pnl, current_account_value, 0.95, min_observations=20)
    var_99 = historical_var(observed_daily_pnl, current_account_value, 0.99, min_observations=100)
    hist_vol = historical_volatility(observed_daily_pnl, current_account_value, annualization_days=365)

    alpha_annual = beta = sharpe_annual = float("nan")
    btc_sample_n = 0
    btc_df = pd.DataFrame()
    try:
        starts = [index.min() for index in [day_index, live_account_daily.index] if len(index)]
        if starts:
            start_ms = int(min(starts).timestamp() * 1000)
            end_ms = int(pd.Timestamp.now(tz="UTC").timestamp() * 1000)
            btc_df = fetch_btc_candles(start_ms, end_ms, interval="1d")
            if not btc_df.empty:
                btc_df["day"] = pd.to_datetime(btc_df["t"], unit="ms", utc=True).dt.tz_localize(None).dt.normalize()
                btc_close = btc_df.assign(close=btc_df["c"].astype(float)).set_index("day")["close"].sort_index()
                btc_returns = btc_close.pct_change()
                equity = daily_account.reindex(net_change.index).ffill()
                portfolio_returns = net_change / equity.shift(1)
                compare = pd.concat([
                    portfolio_returns.rename("pnl"),
                    btc_returns.reindex(portfolio_returns.index).rename("btc"),
                ], axis=1).dropna()
                btc_sample_n = len(compare)
                if btc_sample_n >= 2 and compare["btc"].var() > 0:
                    beta = compare["pnl"].cov(compare["btc"]) / compare["btc"].var()
                    alpha_annual = (compare["pnl"].mean() - beta * compare["btc"].mean()) * 365 * 100
                    pnl_std = compare["pnl"].std()
                    sharpe_annual = compare["pnl"].mean() / pnl_std * (365 ** 0.5) if pnl_std else float("nan")
    except Exception as exc:
        print(f"BTC metrics unavailable: {type(exc).__name__}")

    return SimpleNamespace(**{
        "account_series": account_series, "daily_account": daily_account,
        "realized_series": realized_series, "daily_realized": daily_realized,
        "daily_unrealized": daily_unrealized, "day_index": day_index,
        "realized_avg_day": realized_avg_day, "u_avg_day": u_avg_day,
        "net_change": net_change, "net_avg_day": net_avg_day,
        "daily_return_pct": daily_return_pct, "avg_daily_return_pct": avg_daily_return_pct,
        "max_daily_gain_pct": max_daily_gain_pct, "max_daily_loss_pct": max_daily_loss_pct,
        "monthly_return_pct": monthly_return_pct, "cumulative_pnl": cumulative_pnl,
        "pnl_dd_usd": pnl_dd_usd, "account_dd_usd": account_dd_usd,
        "account_dd_pct": account_dd_pct, "live_account": live_account,
        "btc_df": btc_df, "alpha_annual": alpha_annual, "beta": beta,
        "sharpe_annual": sharpe_annual, "btc_sample_n": btc_sample_n,
        "current_account_value": current_account_value,
        "unrealized_pnl_total": unrealized_pnl_total,
        "observed_daily_pnl": observed_daily_pnl, "var_95": var_95, "var_99": var_99,
        "historical_volatility": hist_vol,
    })


def print_account_metrics(metrics, csv_path, new_api_rows, fill_count):
    """Print the account summary separately from the calculations."""
    account = metrics.daily_account
    print("=== PNL / ACCT ===")
    print(f"Acct value: ${account.iloc[-1]:,.2f}" if not account.empty else "Acct value: n/a")
    print(f"1D: {account.pct_change().iloc[-1] * 100:+.2f}%" if len(account) > 1 else "1D: need 2 daily values")
    print(f"Acct Max DD: ${metrics.account_dd_usd:,.2f} ({metrics.account_dd_pct:.2f}%)" if pd.notna(metrics.account_dd_usd) else "Acct Max DD: n/a")
    print(f"PnL Max DD: ${metrics.pnl_dd_usd:,.2f}" if pd.notna(metrics.pnl_dd_usd) else "PnL Max DD: n/a")
    realized_today = metrics.daily_realized.iloc[-1] if not metrics.daily_realized.empty else 0.0
    valid_unrealized = metrics.daily_unrealized.dropna()
    u_change_today = valid_unrealized.diff().iloc[-1] if len(valid_unrealized) > 1 else float("nan")
    if pd.notna(metrics.realized_avg_day):
        print(f"Realized 1D: ${realized_today:+,.2f} | Avg/d: ${metrics.realized_avg_day:+,.2f}")
    else:
        print(f"Realized 1D: ${realized_today:+,.2f} | Avg/d: n/a")
    if pd.notna(metrics.u_avg_day) and pd.notna(u_change_today):
        print(f"uPnL: ${metrics.unrealized_pnl_total:,.2f} | 1D Δ: ${u_change_today:+,.2f} | Avg/d Δ: ${metrics.u_avg_day:+,.2f}")
    else:
        print(f"uPnL: ${metrics.unrealized_pnl_total:,.2f} | Avg/d Δ: n/a")
    if pd.notna(metrics.avg_daily_return_pct):
        print(f"Avg daily change: ${metrics.net_avg_day:+,.2f} ({metrics.avg_daily_return_pct:+.2f}%) | Max daily gain: {metrics.max_daily_gain_pct:+.2f}% | Max daily loss: {metrics.max_daily_loss_pct:+.2f}%")
    else:
        print("Daily return stats: need matched daily PnL and account values")
    if pd.notna(metrics.beta) and pd.notna(metrics.sharpe_annual):
        print(f"BTC α: {metrics.alpha_annual:+.2f}%/yr | β: {metrics.beta:.2f} | Sharpe: {metrics.sharpe_annual:.2f} | n={metrics.btc_sample_n}")
    else:
        print(f"BTC α / β / Sharpe: need more matched daily data (n={metrics.btc_sample_n})")

    for confidence, result, minimum in [(95, metrics.var_95, 20), (99, metrics.var_99, 100)]:
        if pd.notna(result["usd"]):
            print(f"1D Hist. VaR {confidence}%: ${result['usd']:,.2f} ({result['pct']:.2f}% of current value) | n={result['n']}")
        else:
            print(f"1D Hist. VaR {confidence}%: n/a (need {minimum} observations; n={result['n']})")
    hist_vol = metrics.historical_volatility
    if pd.notna(hist_vol["annualized_pct"]):
        print(f"Hist. vol: {hist_vol['daily_pct']:.2f}%/day | {hist_vol['annualized_pct']:.2f}%/yr (365d) | n={hist_vol['n']}")
    else:
        print(f"Hist. vol: n/a (need 2 observations; n={hist_vol['n']})")
    print("VaR and volatility use observed daily realized spot/perp PnL and recorded daily uPnL changes, scaled by current account value; unobserved historical mark-to-market days are excluded.")
    print(f"CSV: {csv_path.resolve()} | New API rows: {new_api_rows}")
    if fill_count >= 10000:
        print("Realized history limited to latest 10,000 fills.")


account_metrics = calculate_account_metrics(chart_df, current_account_value, unrealized_pnl_total)
print_account_metrics(account_metrics, CSV_PATH, history_rows_added, len(realized_fills))


## Account charts

Plots account value, daily realized and unrealized PnL, cumulative PnL, and monthly return distribution.


In [ ]:
"""Render the account and PnL dashboard."""
def plot_account_dashboard(metrics):
    """Plot account value, daily PnL, cumulative PnL, and monthly return histogram."""
    fig = plt.figure(figsize=(12, 14))
    grid = fig.add_gridspec(4, 1)
    ax_acct = fig.add_subplot(grid[0, 0])
    ax_pnl = fig.add_subplot(grid[1, 0], sharex=ax_acct)
    ax_cum = fig.add_subplot(grid[2, 0], sharex=ax_acct)
    ax_hist = fig.add_subplot(grid[3, 0])

    if not metrics.daily_account.empty:
        ax_acct.plot(metrics.daily_account.index, metrics.daily_account.values, linewidth=2, color="#6c757d", label="Account Value")
        ax_acct.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
    ax_acct.grid(alpha=0.3)
    if not metrics.daily_realized.empty:
        ax_pnl.plot(metrics.day_index, metrics.daily_realized.values, linewidth=2, marker="o", markersize=3, color="#457b9d", label="Realized")
    if not metrics.daily_unrealized.empty:
        ax_pnl.plot(metrics.day_index, metrics.daily_unrealized.values, linewidth=2, marker="o", markersize=3, color="#e76f51", label="Unrealized")
    ax_pnl.axhline(0, color="#777", linewidth=0.8)
    ax_pnl.grid(alpha=0.3)
    if not metrics.daily_realized.empty or not metrics.daily_unrealized.empty:
        ax_pnl.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
    if not metrics.cumulative_pnl.empty:
        ax_cum.plot(metrics.cumulative_pnl.index, metrics.cumulative_pnl.values, linewidth=2, color="#8e44ad", label="Cumulative PnL")
        ax_cum.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
    ax_cum.axhline(0, color="#777", linewidth=0.8)
    ax_cum.grid(alpha=0.3)

    # Keep impossible returns out of the display, without changing any metrics.
    returns = metrics.monthly_return_pct
    if not returns.empty:
        display_returns = returns[returns >= -100]
        excluded = len(returns) - len(display_returns)
        if not display_returns.empty:
            step = 5.0
            lower = np.floor(display_returns.min() / step) * step
            upper = np.ceil(display_returns.max() / step) * step
            if lower == upper:
                lower, upper = lower - step, upper + step
            bins = np.arange(lower, upper + step * 1.01, step)
            ax_hist.hist(display_returns.values, bins=bins, color="#457b9d", alpha=0.8, label="Monthly returns")
            ax_hist.axvline(display_returns.mean(), color="#e76f51", linestyle="--", linewidth=2, label="Average shown")
            ax_hist.axvline(0, color="#555", linewidth=0.8)
            ax_hist.set_xlim(lower, upper)
            ax_hist.set_xticks(np.arange(lower, upper + step * 0.5, step))
            ax_hist.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
            if excluded:
                print(f"Monthly return chart omits {excluded} return(s) below -100%; other data and stats are unchanged.")
        else:
            ax_hist.text(0.5, 0.5, "No valid monthly returns to display", ha="center", va="center", transform=ax_hist.transAxes)
    else:
        ax_hist.text(0.5, 0.5, "Need matched monthly return observations", ha="center", va="center", transform=ax_hist.transAxes)
    ax_hist.set_title("Monthly Return Distribution")
    ax_hist.set_xlabel("Monthly return (%)")
    ax_hist.set_ylabel("Months")
    ax_hist.grid(alpha=0.3)
    fig.tight_layout(rect=[0, 0, 0.82, 1])
    plt.show()


plot_account_dashboard(account_metrics)


## Account vs BTC

Shows the intraday live-snapshot index and monthly lifetime realized-PnL reconstruction.


In [ ]:
"""Render intraday and lifetime account-vs-BTC indexes."""
def plot_value_added_indexes(metrics, chart_df, current_account_value):
    """Plot the live intraday index and monthly realized-PnL lifetime proxy."""
    live_account = metrics.live_account
    btc_daily = metrics.btc_df

    # Use 15-minute candles for intraday snapshots so BTC changes within the day.
    intraday_candles = pd.DataFrame()
    if not live_account.empty:
        try:
            start_ms = int((live_account.index.min() - pd.Timedelta(days=1)).timestamp() * 1000)
            end_ms = int(pd.Timestamp.now(tz="UTC").timestamp() * 1000)
            intraday_candles = fetch_btc_candles(start_ms, end_ms, interval="15m")
        except Exception as exc:
            print(f"Value Added Index BTC data unavailable: {type(exc).__name__}: {exc}")

    if not live_account.empty and not intraday_candles.empty:
        account_points = live_account.groupby(level=0).last().sort_index()
        btc_points = pd.DataFrame({
            "time": pd.to_datetime(intraday_candles["T"], unit="ms", utc=True).dt.tz_localize(None).astype("datetime64[ns]"),
            "BTC": intraday_candles["c"].astype(float),
        }).dropna().sort_values("time").drop_duplicates("time", keep="last")
        account_points = account_points.rename("Account").rename_axis("time").reset_index()
        account_points["time"] = pd.to_datetime(account_points["time"]).astype("datetime64[ns]")
        matched = pd.merge_asof(account_points.sort_values("time"), btc_points, on="time", direction="backward")
        matched = matched.dropna(subset=["BTC"])
        matched = matched[(matched["Account"] != 0) & (matched["BTC"] != 0)]
        if not matched.empty:
            index = matched.set_index("time")[["Account", "BTC"]]
            index = index.div(index.iloc[0]).mul(100)
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(index.index, index["Account"], linewidth=2, marker="o", markersize=4, label="Account")
            ax.plot(index.index, index["BTC"], linewidth=2, marker="o", markersize=4, label="BTC (15m)")
            ax.axhline(100, color="#777", linewidth=0.8, linestyle="--")
            ax.set_title("Value Added Index ($100 start): Account vs BTC")
            ax.set_ylabel("Value of $100")
            ax.grid(alpha=0.3)
            ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
            fig.tight_layout(rect=[0, 0, 0.82, 1])
            plt.show()
        else:
            print("Value Added Index unavailable: no finished BTC candles match account snapshots.")
    else:
        print("Value Added Index unavailable: need account snapshots and BTC 15-minute candle data.")

    # Reconstruct the long-range equity proxy backwards from today's live value.
    perp_pnl = chart_df.loc[chart_df["source"] == "perp_realized_daily", ["ts", "realized_pnl"]].dropna()
    spot_pnl = chart_df.loc[chart_df["source"] == "spot_realized_daily", ["ts", "realized_pnl"]].dropna()
    funding_pnl = chart_df.loc[chart_df["source"] == "funding_daily", ["ts", "realized_pnl"]].dropna()
    realized = pd.concat([perp_pnl, spot_pnl, funding_pnl], ignore_index=True)
    if realized.empty or current_account_value <= 0 or btc_daily.empty:
        print("Lifetime index unavailable: need saved realized PnL, positive account value, and BTC daily candles.")
        return

    realized["day"] = pd.to_datetime(realized["ts"]).dt.normalize()
    daily_pnl = realized.groupby("day")["realized_pnl"].sum().sort_index()
    days = pd.date_range(daily_pnl.index.min(), pd.Timestamp.now(tz="UTC").tz_localize(None).normalize(), freq="1D")
    cumulative = daily_pnl.reindex(days, fill_value=0.0).cumsum()
    reconstructed = current_account_value - (cumulative.iloc[-1] - cumulative)
    monthly = reconstructed.resample("ME").last().dropna()
    monthly = monthly[monthly > 0]
    if monthly.empty:
        print("Lifetime index unavailable: reconstructed historical account values are not positive.")
        return

    account_monthly = monthly.rename("Account (realized PnL)").rename_axis("time").reset_index()
    account_monthly["time"] = pd.to_datetime(account_monthly["time"]).astype("datetime64[ns]")
    btc_monthly = pd.DataFrame({
        "time": pd.to_datetime(btc_daily["T"], unit="ms", utc=True).dt.tz_localize(None).astype("datetime64[ns]"),
        "BTC": btc_daily["c"].astype(float),
    }).dropna().sort_values("time").drop_duplicates("time", keep="last")
    matched = pd.merge_asof(account_monthly.sort_values("time"), btc_monthly, on="time", direction="backward")
    matched = matched.dropna(subset=["BTC"])
    matched = matched[(matched["Account (realized PnL)"] > 0) & (matched["BTC"] > 0)]
    if matched.empty:
        print("Lifetime index unavailable: no overlapping reconstructed account values and BTC candles.")
        return

    index = matched.set_index("time")[["Account (realized PnL)", "BTC"]]
    index = index.div(index.iloc[0]).mul(100)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(index.index, index["Account (realized PnL)"], linewidth=2, marker="o", markersize=4, label="Account (realized PnL, monthly)")
    ax.plot(index.index, index["BTC"], linewidth=2, marker="o", markersize=4, label="BTC (monthly points)")
    ax.axhline(100, color="#777", linewidth=0.8, linestyle="--")
    ax.set_title("Lifetime Value Added Index ($100 start): Realized PnL vs BTC")
    ax.set_ylabel("Value of $100")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
    fig.tight_layout(rect=[0, 0, 0.82, 1])
    plt.show()
    print("Lifetime account series is anchored to current account value; it includes fee-net realized trades and perp funding, but excludes historical cash flows and unrealized PnL.")


plot_value_added_indexes(account_metrics, chart_df, current_account_value)


## Open positions

Displays positions across perp DEXes and summarizes value, uPnL, margin, leverage, and liquidation distance.


In [ ]:
"""Fetch, format, and summarize open positions."""
def fetch_position_rows(user=ADDR):
    """Return all open positions across the main perp DEX and HIP-3 DEXes."""
    states = fetch_perp_states(user)
    positions = [
        (dex or "Hyperliquid", item)
        for dex, state in states.items()
        for item in state.get("assetPositions", [])
    ]
    rows = []
    for dex_name, item in positions:
        position = item["position"]
        size = as_float(position.get("szi"))
        value = as_float(position.get("positionValue"))
        mark = abs(value) / abs(size) if size else 0.0
        liq_price = as_float(position.get("liquidationPx"))
        if liq_price > 0 and mark > 0:
            liq_distance = (mark - liq_price) / mark * 100 if size > 0 else (liq_price - mark) / mark * 100
        else:
            liq_distance = float("nan")
        funding = -as_float((position.get("cumFunding") or {}).get("sinceOpen"))
        margin = as_float(position.get("marginUsed"))
        rows.append({
            "Coin": position["coin"], "Sz": size,
            "Side": "LONG" if size > 0 else "SHORT",
            "Entry": as_float(position.get("entryPx")), "Mark": mark, "Pos": value,
            "Liq%": liq_distance, "uPnL": as_float(position.get("unrealizedPnl")),
            "ROE": as_float(position.get("returnOnEquity")) * 100,
            "Lev.": abs(value) / margin if margin else float("nan"),
            "Liq Px": liq_price, "Margin": margin, "Funding": funding,
        })
    return states, rows


def display_positions(rows):
    """Render the position table, sorted by absolute notional value."""
    if not rows:
        print("No open positions.")
        return
    frame = pd.DataFrame(rows).sort_values("Pos", key=lambda values: values.abs(), ascending=False).reset_index(drop=True)
    usd = lambda value: f"-${abs(value):,.2f}" if value < 0 else f"${value:,.2f}"
    styler = frame.style
    color_map = getattr(styler, "map", None) or styler.applymap
    styled = color_map(
        lambda value: "color: #2a9d8f" if isinstance(value, (int, float)) and value > 0
        else ("color: #e63946" if isinstance(value, (int, float)) and value < 0 else ""),
        subset=["uPnL", "ROE", "Funding"],
    )
    liq_style = lambda value: "" if pd.isna(value) else ("color: #e63946" if value <= 10 else ("color: #d97706" if value <= 25 else "color: #2a9d8f"))
    styled = styled.map(liq_style, subset=["Liq%"]) if hasattr(styled, "map") else styled.applymap(liq_style, subset=["Liq%"])
    display(styled.format({
        "Sz": "{:,.2f}", "Entry": "{:,.2f}", "Mark": "{:,.2f}", "Pos": usd,
        "Liq%": "{:,.2f}%", "uPnL": usd, "ROE": "{:,.2f}%", "Lev.": "{:,.2f}x",
        "Liq Px": "{:,.2f}", "Margin": usd, "Funding": usd,
    }))


def summarize_positions(rows, user=ADDR):
    """Calculate total perp exposure and account-wide margin statistics."""
    position_value = sum(abs(row["Pos"]) for row in rows)
    unrealized_pnl = sum(row["uPnL"] for row in rows)
    margin_used = sum(row["Margin"] for row in rows)
    funding = sum(row["Funding"] for row in rows)
    balance, priced, excluded = spot_balance_usd(user)
    return SimpleNamespace(
        position_value=position_value, unrealized_pnl=unrealized_pnl,
        margin_used=margin_used, funding=funding, balance=balance,
        current_account_value=balance + unrealized_pnl,
        roe=unrealized_pnl / margin_used * 100 if margin_used else 0.0,
        leverage=position_value / margin_used if margin_used else float("nan"),
        margin_pct=margin_used / balance * 100 if balance else float("nan"),
        priced_spot=priced, unpriced_spot=excluded,
    )


states, rows = fetch_position_rows(ADDR)
display_positions(rows)
position_summary = summarize_positions(rows, ADDR)
position_value_total = position_summary.position_value
unrealized_pnl_total = position_summary.unrealized_pnl
margin_used_total = position_summary.margin_used
funding_total = position_summary.funding
balance_total = position_summary.balance
current_account_value = position_summary.current_account_value
priced_spot = position_summary.priced_spot
unpriced_spot = position_summary.unpriced_spot
print("\n=== POSITIONS ===")
print(f"Pos: ${position_value_total:,.2f}")
print(f"uPnL: ${unrealized_pnl_total:,.2f}")
print(f"ROE: {position_summary.roe:,.2f}%")
print(f"Margin: ${margin_used_total:,.2f}")
print(f"Margin % bal: {position_summary.margin_pct:,.2f}%" if pd.notna(position_summary.margin_pct) else "Margin % bal: n/a")
print(f"Lev. (Pos / Margin): {position_summary.leverage:,.2f}x" if pd.notna(position_summary.leverage) else "Lev.: n/a")
print(f"Funding: ${funding_total:,.2f}")


## Realized PnL statistics

Reports fee-net spot/perp realized results with signed perp funding, wins/losses, expected value, and liquidation stats.


In [ ]:
"""Calculate and display realized PnL statistics."""
def split_realized_fills(realized_df):
    """Separate the normalized fill table into spot and perp frames."""
    if realized_df.empty:
        empty = pd.DataFrame()
        return empty.copy(), empty.copy()
    return (
        realized_df[realized_df["assetType"] == "perp"].copy(),
        realized_df[realized_df["assetType"] == "spot"].copy(),
    )


def realized_pnl_stats(frame, funding_history=None, now=None):
    """Summarize fee-net fill PnL plus signed funding, with trade stats based on fills."""
    now = now or pd.Timestamp.now(tz="UTC")
    quarter_start_month = ((now.month - 1) // 3) * 3 + 1
    quarter_start = pd.Timestamp(year=now.year, month=quarter_start_month, day=1, tz="UTC")
    year_start = pd.Timestamp(year=now.year, month=1, day=1, tz="UTC")
    if frame.empty:
        frame = pd.DataFrame(columns=["closedPnl", "notional", "ts"])
    if funding_history is None or funding_history.empty:
        funding_history = pd.DataFrame(columns=["fundingPnl", "ts"])
    else:
        funding_history = funding_history.copy()
        funding_history["ts"] = pd.to_datetime(funding_history["ts"], utc=True)

    empty_result = {
        "total": 0.0, "funding": 0.0, "1d": 0.0, "7d": 0.0, "30d": 0.0, "qtd": 0.0, "ytd": 0.0,
        "wins": 0, "losses": 0, "win_p": float("nan"), "loss_p": float("nan"),
        "avg_win": float("nan"), "avg_loss": float("nan"),
        "avg_win_pct": float("nan"), "avg_loss_pct": float("nan"), "ev": float("nan"), "fills": 0,
    }
    if frame.empty and funding_history.empty:
        return empty_result

    def combined_pnl_since(start_time=None):
        fill_part = frame
        funding_part = funding_history
        if start_time is not None:
            if not fill_part.empty:
                fill_part = fill_part.loc[fill_part["ts"] >= start_time]
            if not funding_part.empty:
                funding_part = funding_part.loc[funding_part["ts"] >= start_time]
        fill_pnl = fill_part["closedPnl"].sum() if not fill_part.empty else 0.0
        funding_pnl = funding_part["fundingPnl"].sum() if not funding_part.empty else 0.0
        return float(fill_pnl + funding_pnl)

    wins = frame.loc[frame["closedPnl"] > 0]
    losses = frame.loc[frame["closedPnl"] < 0]
    win_returns = (wins["closedPnl"] / wins["notional"] * 100).replace([float("inf"), -float("inf")], float("nan")).dropna()
    loss_returns = (losses["closedPnl"] / losses["notional"] * 100).replace([float("inf"), -float("inf")], float("nan")).dropna()
    closed_count = len(wins) + len(losses)
    win_probability = len(wins) / closed_count if closed_count else float("nan")
    loss_probability = len(losses) / closed_count if closed_count else float("nan")
    average_win = wins["closedPnl"].mean() if not wins.empty else 0.0
    average_loss = losses["closedPnl"].mean() if not losses.empty else 0.0
    expected_value = win_probability * average_win + loss_probability * average_loss if closed_count else float("nan")
    return {
        "total": combined_pnl_since(),
        "funding": float(funding_history["fundingPnl"].sum()) if not funding_history.empty else 0.0,
        "1d": combined_pnl_since(now - pd.Timedelta(days=1)),
        "7d": combined_pnl_since(now - pd.Timedelta(days=7)),
        "30d": combined_pnl_since(now - pd.Timedelta(days=30)),
        "qtd": combined_pnl_since(quarter_start),
        "ytd": combined_pnl_since(year_start),
        "wins": len(wins), "losses": len(losses), "win_p": win_probability, "loss_p": loss_probability,
        "avg_win": average_win, "avg_loss": average_loss,
        "avg_win_pct": win_returns.mean() if not win_returns.empty else float("nan"),
        "avg_loss_pct": loss_returns.mean() if not loss_returns.empty else float("nan"),
        "ev": expected_value, "fills": len(frame),
    }


def liquidation_stats(perp_frame, funding_history=None):
    """Count liquidation closes and calculate their share of closes and perp PnL."""
    if perp_frame.empty:
        return {"liquidations": 0, "closes": 0, "close_pct": 0.0, "pnl": 0.0, "pnl_pct": float("nan")}
    directions = perp_frame["dir"].fillna("").astype(str).str.lower()
    close_mask = directions.str.contains("close|liquidat") | (perp_frame["closedPnl"] != 0)
    closes = perp_frame.loc[close_mask].drop_duplicates(subset=["time", "coin", "dir"])
    liquidations = perp_frame.loc[directions.str.contains("liquidat")].drop_duplicates(subset=["time", "coin", "dir"])
    funding_pnl = funding_history["fundingPnl"].sum() if funding_history is not None and not funding_history.empty else 0.0
    total_pnl = perp_frame["closedPnl"].sum() + funding_pnl
    liquidation_pnl = liquidations["closedPnl"].sum()
    return {
        "liquidations": len(liquidations), "closes": len(closes),
        "close_pct": len(liquidations) / len(closes) * 100 if len(closes) else 0.0,
        "pnl": liquidation_pnl,
        "pnl_pct": liquidation_pnl / total_pnl * 100 if total_pnl else float("nan"),
    }


def print_realized_stats(frame, label, funding_history=None):
    """Print one spot or perp realized PnL summary."""
    stats = realized_pnl_stats(frame, funding_history=funding_history)
    print(f"\n=== REALIZED {label} ===")
    funding_label = f" | Funding: ${stats['funding']:+,.2f}" if label == "PERP" else ""
    print(f"PnL: ${stats['total']:,.2f}{funding_label} | 1D: ${stats['1d']:,.2f} | QTD: ${stats['qtd']:,.2f} | YTD: ${stats['ytd']:,.2f} | Since inc.: ${stats['total']:,.2f}")
    print(f"W: {stats['wins']} ({stats['win_p']:.1%}) | Avg W: ${stats['avg_win']:,.2f} ({stats['avg_win_pct']:+.2f}% ntl)")
    print(f"L: {stats['losses']} ({stats['loss_p']:.1%}) | Avg L: ${stats['avg_loss']:,.2f} ({stats['avg_loss_pct']:+.2f}% ntl)")
    print(f"EV/close: ${stats['ev']:,.2f}" if pd.notna(stats["ev"]) else "EV/close: n/a")


def perp_pnl_breakdown(perp_fills, funding_history=None):
    """Break perp net PnL into gross trading, fee cost, and funding contributions.

    Hyperliquid reports fill fees as positive amounts, while closedPnl already
    reflects those fees. Therefore gross trading PnL is closedPnl plus fees.
    Dollar reconciliation is only valid when all reported perp fees are USDC.
    """
    closed_pnl = float(perp_fills["closedPnl"].sum()) if not perp_fills.empty else 0.0
    funding_pnl = (
        float(funding_history["fundingPnl"].sum())
        if funding_history is not None and not funding_history.empty else 0.0
    )
    net_pnl = closed_pnl + funding_pnl

    fee_rows = perp_fills.dropna(subset=["fee"]).copy() if "fee" in perp_fills else pd.DataFrame()
    if not fee_rows.empty:
        fee_rows["feeToken"] = fee_rows["feeToken"].fillna("Unknown").astype(str).str.strip()
        fee_rows.loc[fee_rows["feeToken"] == "", "feeToken"] = "Unknown"
        nonzero_tokens = set(fee_rows.loc[fee_rows["fee"].abs() > 1e-12, "feeToken"])
    else:
        nonzero_tokens = set()
    fees_are_usd = not nonzero_tokens or nonzero_tokens <= {"USDC"}

    if fees_are_usd:
        fees_paid = float(fee_rows["fee"].sum()) if not fee_rows.empty else 0.0
        rows = [
            ("Trading", closed_pnl + fees_paid),
            ("Fees", -fees_paid),
            ("Funding", funding_pnl),
            ("Net perp PnL", net_pnl),
        ]
        data = pd.DataFrame(rows, columns=["Component", "Amount ($)"])
        data["% of net PnL"] = (
            data["Amount ($)"] / net_pnl * 100 if abs(net_pnl) > 1e-12 else float("nan")
        )
        return data, None

    data = pd.DataFrame([
        ("Trading", float("nan")), ("Fees", float("nan")),
        ("Funding", funding_pnl), ("Net perp PnL", net_pnl),
    ], columns=["Component", "Amount ($)"])
    data["% of net PnL"] = (
        data["Amount ($)"] / net_pnl * 100 if abs(net_pnl) > 1e-12 else float("nan")
    )
    return data, ", ".join(sorted(nonzero_tokens))


perp_df, spot_df = split_realized_fills(realized_df)
print_realized_stats(spot_df, "SPOT")
print_realized_stats(perp_df, "PERP", funding_history=funding_history_df)
print("\n=== PERP PNL BREAKDOWN ===")
perp_breakdown, non_usd_fee_tokens = perp_pnl_breakdown(perp_df, funding_history_df)
display(perp_breakdown.style.format({
    "Amount ($)": "${:,.2f}", "% of net PnL": "{:+.2f}%",
}, na_rep="n/a"))
if non_usd_fee_tokens:
    print(
        "Trading and fee dollar contributions are n/a because fee amounts use "
        f"non-USDC token(s): {non_usd_fee_tokens}. Net PnL still reconciles from "
        "closedPnl plus signed funding."
    )
else:
    print("Trading is gross of fees; fees are shown as a negative contribution. Shares use net realized perp PnL as denominator.")
liquidations = liquidation_stats(perp_df, funding_history=funding_history_df)
if pd.notna(liquidations["pnl_pct"]):
    print(f"Liq: {liquidations['liquidations']}/{liquidations['closes']} closes ({liquidations['close_pct']:.2f}%) | Liq PnL: ${liquidations['pnl']:,.2f} ({liquidations['pnl_pct']:.2f}% of perp PnL)")
else:
    print(f"Liq: {liquidations['liquidations']}/{liquidations['closes']} closes ({liquidations['close_pct']:.2f}%) | Liq PnL: ${liquidations['pnl']:,.2f} (share n/a)")
def print_total_fees(fills_frame):
    """Show fees from fetched fills, grouped by the token Hyperliquid charged."""
    print("\n=== FEES PAID ===")
    if fills_frame.empty or "fee" not in fills_frame.columns:
        print("Fees: n/a (no fee data in the available fills)")
        return
    fee_rows = fills_frame.dropna(subset=["fee"]).copy()
    if fee_rows.empty:
        print("Fees: n/a (no fee data in the available fills)")
        return
    fee_rows["feeToken"] = fee_rows["feeToken"].fillna("Unknown").astype(str).str.strip()
    fee_rows.loc[fee_rows["feeToken"] == "", "feeToken"] = "Unknown"
    fee_totals = fee_rows.groupby("feeToken")["fee"].sum().sort_index()
    formatted = []
    for token, amount in fee_totals.items():
        if token == "USDC":
            formatted.append(f"${amount:,.2f} USDC")
        else:
            formatted.append(f"{amount:,.8f} {token}")
    print(f"Total: {' + '.join(formatted)}")
    print("Scope: fees in the available fill history (up to 10,000 fills); amounts are grouped by fee token.")


print_total_fees(realized_df)
print("Realized PnL uses fill closedPnl (trading fees included) plus signed perp funding; funding paid is negative.")
print("Fill PnL, fees, win/loss, and liquidation stats use available fills (API cap: 10,000); funding history is stored separately.")


## Traded volume vs PnL

Compares daily traded notional with fee-net realized fill PnL for perps and spot, including turnover efficiency, correlations, and a scatter plot.

In [ ]:
"""Compare traded turnover with realized fill PnL by market type."""
def build_daily_volume_pnl(fills):
    """Aggregate fill notional and realized PnL by UTC day and asset type."""
    if fills.empty:
        return pd.DataFrame(columns=["day", "assetType", "volume", "realized_pnl", "fills"])
    data = fills.copy()
    data["day"] = pd.to_datetime(data["day"], errors="coerce", utc=True).dt.tz_localize(None)
    data["notional"] = pd.to_numeric(data["notional"], errors="coerce")
    data["closedPnl"] = pd.to_numeric(data["closedPnl"], errors="coerce")
    data = data.dropna(subset=["day", "assetType"])
    return (
        data.groupby(["day", "assetType"], as_index=False)
        .agg(volume=("notional", "sum"), realized_pnl=("closedPnl", "sum"), fills=("closedPnl", "size"))
        .sort_values("day")
        .reset_index(drop=True)
    )


def summarize_volume_pnl(daily_data):
    """Summarize turnover, fill PnL, PnL per turnover, and daily correlations."""
    rows = []
    labels = {"perp": "Perps", "spot": "Spot"}
    for asset_type in ["perp", "spot"]:
        part = daily_data.loc[daily_data["assetType"] == asset_type].copy()
        if part.empty:
            continue
        total_volume = part["volume"].sum()
        total_pnl = part["realized_pnl"].sum()
        rows.append({
            "Market": labels.get(asset_type, asset_type.title()),
            "Volume ($)": total_volume,
            "Realized PnL ($)": total_pnl,
            "PnL / $1k volume ($)": total_pnl / total_volume * 1000 if total_volume else float("nan"),
            "Active days": len(part),
            "Daily Pearson r": part["volume"].corr(part["realized_pnl"], method="pearson") if len(part) >= 2 else float("nan"),
            "Daily Spearman r": part["volume"].corr(part["realized_pnl"], method="spearman") if len(part) >= 2 else float("nan"),
        })
    return pd.DataFrame(rows)


daily_volume_pnl = build_daily_volume_pnl(realized_df)
volume_pnl_summary = summarize_volume_pnl(daily_volume_pnl)
print("=== TRADED VOLUME VS REALIZED PNL ===")
if volume_pnl_summary.empty:
    print("No fill volume or realized PnL data is available.")
else:
    display(volume_pnl_summary.style.format({
        "Volume ($)": "${:,.2f}", "Realized PnL ($)": "${:+,.2f}",
        "PnL / $1k volume ($)": "${:+,.2f}", "Daily Pearson r": "{:+.3f}",
        "Daily Spearman r": "{:+.3f}",
    }, na_rep="n/a"))

    market_types = [kind for kind in ["perp", "spot"] if (daily_volume_pnl["assetType"] == kind).any()]
    fig, axes = plt.subplots(1, len(market_types), figsize=(7 * len(market_types), 5), squeeze=False)
    for ax, asset_type in zip(axes[0], market_types):
        part = daily_volume_pnl.loc[daily_volume_pnl["assetType"] == asset_type]
        ax.scatter(part["volume"], part["realized_pnl"], alpha=0.75, s=36)
        ax.axhline(0, color="gray", linewidth=0.8, alpha=0.7)
        ax.set_title(f"{asset_type.title()}: daily volume vs PnL")
        ax.set_xlabel("Daily traded volume ($)")
        ax.set_ylabel("Daily realized PnL ($)")
        ax.grid(True, alpha=0.2)
        if len(part) >= 2 and part["volume"].nunique() > 1:
            slope, intercept = np.polyfit(part["volume"], part["realized_pnl"], 1)
            x_line = np.linspace(part["volume"].min(), part["volume"].max(), 100)
            ax.plot(x_line, slope * x_line + intercept, color="darkorange", linewidth=1.5, label="Linear fit")
            ax.legend(loc="best")
    fig.tight_layout()
    plt.show()

    print("Volume is one-sided fill notional (price × size), counted once per fill. Realized PnL is Hyperliquid closedPnl, which includes trading fees; funding is excluded.")
    print("Correlation describes co-movement, not causation. Daily observations with no fills are omitted.")


## PnL attribution

Breaks available realized PnL and open perp uPnL down by asset.


In [ ]:
"""Build and display realized/unrealized PnL attribution by asset."""
def build_pnl_attribution(realized_df, open_positions, funding_history=None):
    """Combine realized fill PnL and current perp uPnL by market/asset."""
    parts = []
    if funding_history is not None and not funding_history.empty:
        funding = funding_history.groupby("coin", as_index=False)["fundingPnl"].sum()
        funding = funding.rename(columns={"coin": "Asset", "fundingPnl": "Realized"})
        funding["Type"] = "Perp"
        funding["uPnL"] = 0.0
        parts.append(funding[["Type", "Asset", "Realized", "uPnL"]])
    if not realized_df.empty:
        realized = realized_df.groupby(["assetType", "coin"], as_index=False)["closedPnl"].sum()
        realized = realized.rename(columns={"assetType": "Type", "coin": "Asset", "closedPnl": "Realized"})
        realized["Type"] = realized["Type"].str.title()
        realized["uPnL"] = 0.0
        parts.append(realized[["Type", "Asset", "Realized", "uPnL"]])
    if open_positions:
        unrealized = pd.DataFrame(open_positions).groupby("Coin", as_index=False)["uPnL"].sum()
        unrealized = unrealized.rename(columns={"Coin": "Asset"})
        unrealized["Type"] = "Perp"
        unrealized["Realized"] = 0.0
        parts.append(unrealized[["Type", "Asset", "Realized", "uPnL"]])
    if not parts:
        return pd.DataFrame(), 0.0

    attribution = pd.concat(parts, ignore_index=True).groupby(
        ["Type", "Asset"], as_index=False
    )[["Realized", "uPnL"]].sum()
    attribution = attribution[(attribution["Realized"] != 0) | (attribution["uPnL"] != 0)]
    if attribution["Asset"].astype(str).str.startswith("@").any():
        try:
            metadata = post({"type": "spotMeta"})
            names = {
                f"@{market.get('index', index)}": market.get("name", f"@{market.get('index', index)}")
                for index, market in enumerate(metadata.get("universe", []))
            }
            attribution["Asset"] = attribution["Asset"].map(lambda asset: names.get(asset, asset))
        except Exception as exc:
            print(f"Spot labels remain as API IDs: {type(exc).__name__}")

    attribution["Total PnL"] = attribution["Realized"] + attribution["uPnL"]
    net_pnl = attribution["Total PnL"].sum()
    attribution["% Net"] = attribution["Total PnL"] / net_pnl * 100 if abs(net_pnl) > 1e-12 else float("nan")
    attribution["_abs_pnl"] = attribution["Total PnL"].abs()
    attribution = attribution.sort_values("_abs_pnl", ascending=False).drop(columns="_abs_pnl").reset_index(drop=True)
    return attribution, net_pnl


print("=== PNL ATTRIBUTION ===")
pnl_attribution, net_attributed_pnl = build_pnl_attribution(realized_df, rows, funding_history=funding_history_df)
if pnl_attribution.empty:
    print("No realized or open-position PnL to attribute.")
else:
    display(pnl_attribution.style.format({
        "Realized": "${:,.2f}", "uPnL": "${:,.2f}",
        "Total PnL": "${:,.2f}", "% Net": "{:+.2f}%",
    }))
    print(f"Net attributed PnL: ${net_attributed_pnl:,.2f} | % Net is signed, so losing assets offset winning assets and shares sum to 100%.")
    print("Scope: available realized fills (API cap: 10,000) + saved perp funding + current open perp uPnL; spot unrealized PnL is not included.")


## Portfolio API reference

Fetches the latest allTime API value as a separate reference measure.


In [ ]:
"""Fetch the latest allTime portfolio API value as a separate reference."""
def fetch_latest_portfolio_value(user=ADDR):
    """Return the last allTime accountValueHistory value, or NaN if absent."""
    response = post({"type": "portfolio", "user": user})
    periods = {
        item[0]: item[1]
        for item in response
        if isinstance(item, (list, tuple)) and len(item) == 2
    }
    history = periods.get("allTime", {}).get("accountValueHistory", [])
    return float(history[-1][1]) if history else float("nan")


portfolio_latest = fetch_latest_portfolio_value(ADDR)
